In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

HOSP = Path("../data/mimiciv-3.1/hosp")
PROCESSED = Path("../data/processed")

stays = pd.read_parquet(PROCESSED / "admissions_65plus.parquet")
admissions = pd.read_csv(
    HOSP / "admissions.csv",
    usecols=["subject_id", "hadm_id", "admittime", "dischtime", "admission_type", "admission_location"],
    parse_dates=["admittime", "dischtime"],
)

print("stays:", len(stays))            
print("all admissions:", len(admissions))  

In [ ]:
# filtering only the unplanned stays
unplanned_types = ["EW EMER.", "DIRECT EMER.", "URGENT", "OBSERVATION ADMIT"]
returns = admissions[admissions["admission_type"].isin(unplanned_types)].copy()

# eliminating the transfers from other hospitals since it's same stay
returns = returns[returns["admission_location"] != "TRANSFER FROM HOSPITAL"].copy()

# combine stays and returns
pairs = pd.merge(
    stays[["subject_id", "hadm_id", "dischtime"]],
    returns[["subject_id", "hadm_id", "admittime"]],
    on="subject_id", how="left", suffixes=("", "_ret"),
)
display(pairs.head())

# created the window for 30 days readmission (1 if readmitted in 30 days or less, otherwise 0)
window = pairs[
    (pairs["admittime"] >= pairs["dischtime"]) &
    (pairs["admittime"] <= pairs["dischtime"] + pd.Timedelta(days=30))
]
display(window.head())

# create new column in stays that marks readmission based on the window
stays["readmit_30d"] = stays["hadm_id"].isin(window["hadm_id"]).astype(int)
# filter out the patients that passed away after discharge
died_30d = stays["dod"].notna() & (stays["dod"] <= stays["dischtime"] + pd.Timedelta(days=30))
to_remove = died_30d & (stays["readmit_30d"] == 0)
print("removed (died within 30d, not readmitted):", to_remove.sum())
stays = stays[~to_remove].copy()

display(stays["readmit_30d"].value_counts())
display(stays["readmit_30d"].value_counts(normalize=True))

In [ ]:
rate = stays.groupby("admission_type")["readmit_30d"].mean().sort_values() * 100
overall = stays["readmit_30d"].mean() * 100


fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(rate.index, rate.values, color="steelblue")
ax.axvline(overall, color="red", linestyle="--", label=f"overall = {overall:.1f}%")
ax.set_xlabel("30-day readmission rate (%)")
ax.set_title("Readmission rate by admission type, stays of patients aged 65+")
ax.legend()
ax.grid(axis="x", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

pct = stays["readmit_30d"].value_counts(normalize=True).sort_index() * 100
counts = stays["readmit_30d"].value_counts().sort_index()
names = ["Not readmitted (0)", "Readmitted ≤30 days (1)"]

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(names, pct.values, color=["steelblue", "indianred"])
ax.bar_label(bars, labels=[f"{p:.1f}%\n(n={c:,})" for p, c in zip(pct.values, counts.values)])
ax.set_ylim(0, 100)
ax.set_ylabel("% of stays")
ax.set_title("30-day readmission, stays of patients aged 65+")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
# number every patient's stays 0, 1, ...
adm = admissions.sort_values(["subject_id", "admittime"]).copy()
adm["prior_admissions"] = adm.groupby("subject_id").cumcount()

# put prior admission in stays
stays = pd.merge(
    stays, adm[["hadm_id", "prior_admissions"]],
    on="hadm_id", how="left", validate="one_to_one",
)
print(stays["prior_admissions"].describe())

# histogram
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(stays["prior_admissions"].clip(upper=30), bins=range(0, 32), edgecolor="white", color="steelblue")
ax.set_xlabel("Number of prior admissions (30 = 30 or more)")
ax.set_ylabel("Number of stays")
ax.set_title("Prior admissions per stay, patients aged 65+")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

stays.to_parquet(PROCESSED / "stays_labeled.parquet", index=False)